В цьому домашньому завданні ми знову працюємо з даними з нашого змагання ["Bank Customer Churn Prediction (DLU Course)"](https://www.kaggle.com/t/7c080c5d8ec64364a93cf4e8f880b6a0).

Тут ми побудуємо рішення задачі класифікації з використанням алгоритмів бустингу: XGBoost та LightGBM, а також використаємо бібліотеку HyperOpt для оптимізації гіперпараметрів.

0. Зчитайте дані `train.csv` в змінну `raw_df` та скористайтесь наведеним кодом нижче аби розділити дані на трнувальні та валідаційні і розділити дані на ознаки з матириці Х та цільову змінну. Назви змінних `train_inputs, train_targets, train_inputs, train_targets` можна змінити на ті, які Вам зручно.

  Наведений скрипт - частина отриманого мною скрипта для обробки даних. Ми тут не викнуємо масштабування та обробку категоріальних змінних, бо хочемо це делегувати алгоритмам, які будемо використовувати. Якщо щось не розумієте в наведених скриптах, рекомендую розібратись: навичка читати код - важлива складова роботи в машинному навчанні.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_curve, auc, classification_report
from typing import Tuple, Dict, Any

np.set_printoptions(legacy='1.25')


def split_train_val(df: pd.DataFrame, target_col: str, test_size: float = 0.2, random_state: int = 42) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    Split the dataframe into training and validation sets.

    Args:
        df (pd.DataFrame): The raw dataframe.
        target_col (str): The target column for stratification.
        test_size (float): The proportion of the dataset to include in the validation split.
        random_state (int): Random state for reproducibility.

    Returns:
        Tuple[pd.DataFrame, pd.DataFrame]: Training and validation dataframes.
    """
    train_df, val_df = train_test_split(df, test_size=test_size, random_state=random_state, stratify=df[target_col])
    return train_df, val_df


def separate_inputs_targets(df: pd.DataFrame, input_cols: list, target_col: str) -> Tuple[pd.DataFrame, pd.Series]:
    """
    Separate inputs and targets from the dataframe.

    Args:
        df (pd.DataFrame): The dataframe.
        input_cols (list): List of input columns.
        target_col (str): Target column.

    Returns:
        Tuple[pd.DataFrame, pd.Series]: DataFrame of inputs and Series of targets.
    """
    inputs = df[input_cols].copy()
    targets = df[target_col].copy()
    return inputs, targets


def get_auroc(model, inputs, targets, title):
    preds = model.predict_proba(inputs)[:, 1]
    fpr, tpr, _ = roc_curve(targets, preds)
    roc_auc = auc(fpr, tpr)
    print(f"{title} ROC AUC: {roc_auc:.4f}")

In [ ]:
raw_df = pd.read_csv("drive/MyDrive/Colab Notebooks/data/train.csv")

In [ ]:
raw_df.head(3)

,id,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,0,15779985.0,Nwankwo,678.0,France,Male,29.0,4.0,0.0,3.0,1.0,0.0,180626.36,0.0
1,1,15650086.0,Ch'in,687.0,France,Female,34.0,1.0,0.0,2.0,0.0,1.0,63736.17,0.0
2,2,15733602.0,Thompson,682.0,France,Female,52.0,6.0,0.0,3.0,0.0,0.0,179655.87,1.0


In [ ]:
input_cols = list(raw_df.columns)[1:-1]
target_col = 'Exited'
split_dfs = split_train_val(raw_df, target_col)
train_inputs, train_targets = separate_inputs_targets(split_dfs[0], input_cols, target_col)
val_inputs, val_targets = separate_inputs_targets(split_dfs[1], input_cols, target_col)

1. В тренувальному та валідаційному наборі перетворіть категоріальні ознаки на тип `category`. Можна це зробити двома способами:
 1. `df[col_name].astype('category')`, як було продемонстровано в лекції
 2. використовуючи метод `pd.Categorical(df[col_name])`

In [ ]:
cat_features = train_inputs.select_dtypes(include='object').columns
train_inputs[cat_features] = train_inputs[cat_features].astype('category')
val_inputs[cat_features] = val_inputs[cat_features].astype('category')

In [ ]:
train_inputs.info()

<class 'pandas.core.frame.DataFrame'>
Index: 12000 entries, 7180 to 9360
Data columns (total 12 columns):
 #   Column           Non-Null Count  Dtype   
---  ------           --------------  -----   
 0   CustomerId       12000 non-null  float64 
 1   Surname          12000 non-null  category
 2   CreditScore      12000 non-null  float64 
 3   Geography        12000 non-null  category
 4   Gender           12000 non-null  category
 5   Age              12000 non-null  float64 
 6   Tenure           12000 non-null  float64 
 7   Balance          12000 non-null  float64 
 8   NumOfProducts    12000 non-null  float64 
 9   HasCrCard        12000 non-null  float64 
 10  IsActiveMember   12000 non-null  float64 
 11  EstimatedSalary  12000 non-null  float64 
dtypes: category(3), float64(9)
memory usage: 1006.5 KB


2. Навчіть на отриманих даних модель `XGBoostClassifier`. Параметри алгоритму встановіть на свій розсуд, ми далі будемо їх тюнити. Рекомендую тренувати не дуже складну модель.

  Опис всіх конфігураційних параметрів XGBoostClassifier - тут https://xgboost.readthedocs.io/en/stable/parameter.html#global-config

  **Важливо:** зробіть такі налаштування `XGBoostClassifier` аби він самостійно обробляв незаповнені значення в даних і обробляв категоріальні колонки.

  Можна також, якщо працюєте в Google Colab, увімкнути можливість використання GPU (`Runtime -> Change runtime type -> T4 GPU`) і встановити параметр `device='cuda'` в `XGBoostClassifier` для пришвидшення тренування бустинг моделі.
  
  Після тренування моделі
  1. Виміряйте точність з допомогою AUROC на тренувальному та валідаційному наборах.
  2. Зробіть висновок про отриману модель: вона хороша/погана, чи є high bias/high variance?
  3. Порівняйте якість цієї моделі з тою, що ви отрмали з використанням DecisionTrees раніше. Чи вийшло покращити якість?

In [ ]:
from xgboost import XGBClassifier

In [ ]:
xgb_clf = XGBClassifier(
    max_depth=3,
    n_estimators=10,
    enable_categorical=True,  # для категорійних ознак
    missing=np.nan,           # явне вказування пропущених значень
    device='cuda'
)

xgb_clf.fit(train_inputs, train_targets)

get_auroc(xgb_clf, train_inputs, train_targets, 'Train')
get_auroc(xgb_clf, val_inputs, val_targets, 'Validation')

Train ROC AUC: 0.9455
Validation ROC AUC: 0.9276


This model performs well on both train and validation data.   
There is small gap `0.02` between AUROC results, the model isn’t severely overfitting.

3. Використовуючи бібліотеку `Hyperopt` і приклад пошуку гіперпараметрів для `XGBoostClassifier` з лекції знайдіть оптимальні значення гіперпараметрів `XGBoostClassifier` для нашої задачі. Задайте свою сітку гіперпараметрів виходячи з тих параметрів, які ви б хотіли перебрати. Поставте кількість раундів в підборі гіперпараметрів рівну **20**.

  **Увага!** Для того, аби скористатись hyperopt, нам треба задати функцію `objective`. В ній ми маємо задати loss - це може будь-яка метрика, але бажано використовувтаи ту, яка цільова в вашій задачі. Чим менший лосс - тим ліпша модель на думку hyperopt. Тож, тут нам треба задати loss - негативне значення AUROC. В лекції ми натомість використовували Accuracy.

  Після успішного завершення пошуку оптимальних гіперпараметрів
    - виведіть найкращі значення гіперпараметрів
    - створіть в окремій зміній `final_clf` модель `XGBoostClassifier` з найкращими гіперпараметрами
    - навчіть модель `final_clf`
    - оцініть якість моделі `final_clf` на тренувальній і валідаційній вибірках з допомогою AUROC.
    - зробіть висновок про якість моделі. Чи стала вона краще порівняно з попереднім пунктом (2) цього завдання?

In [ ]:
#!pip install hyperopt

In [ ]:
from hyperopt import fmin, tpe, hp, STATUS_OK, Trials

In [ ]:
def objective(params):
    clf = XGBClassifier(
    n_estimators=int(params['n_estimators']),
        learning_rate=params['learning_rate'],
        max_depth=int(params['max_depth']),
        min_child_weight=params['min_child_weight'], # Мінімальна сума ваг всіх вибірок, необхідна в кінцевому вузлі
        subsample=params['subsample'],               # Частка вибірок, що використовуються для побудови кожного дерева
        colsample_bytree=params['colsample_bytree'], # Частка ознак, що використовуються при побудові кожного дерева
        gamma=params['gamma'],                       # Мінімальне зменшення втрат, необхідне для виконання поділу
        reg_alpha=params['reg_alpha'],               # Параметр регуляризації L1 (Lasso)
        reg_lambda=params['reg_lambda'],             # Параметр регуляризації L2 (Ridge)
        enable_categorical=True,
        use_label_encoder=False,
        missing=np.nan,
        device='cuda',
        early_stopping_rounds=10
    )

    clf.fit(
        train_inputs,
        train_targets,
        eval_set=[(val_inputs, val_targets)],
        verbose=False)
    pred = clf.predict_proba(val_inputs)[:, 1]
    fpr, tpr, _ = roc_curve(val_targets, pred)
    roc_auc = auc(fpr, tpr)

    return {'loss': -roc_auc, 'status': STATUS_OK}

# Простір гіперпараметрів
space = {
    'n_estimators': hp.quniform('n_estimators', 50, 500, 25),
    'learning_rate': hp.uniform('learning_rate', 0.01, 0.3),
    'max_depth': hp.quniform('max_depth', 3, 15, 1),
    'min_child_weight': hp.quniform('min_child_weight', 1, 10, 1),
    'subsample': hp.uniform('subsample', 0.5, 1.0),
    'colsample_bytree': hp.uniform('colsample_bytree', 0.5, 1.0),
    'gamma': hp.uniform('gamma', 0, 0.5),
    'reg_alpha': hp.uniform('reg_alpha', 0, 1),
    'reg_lambda': hp.uniform('reg_lambda', 0, 1)
}

# Оптимізація
trials = Trials()
best = fmin(fn=objective, space=space, algo=tpe.suggest, max_evals=20, trials=trials)

# Перетворення значень гіперпараметрів у кінцеві типи
best['n_estimators'] = int(best['n_estimators'])
best['max_depth'] = int(best['max_depth'])
best['min_child_weight'] = int(best['min_child_weight'])

# Навчання фінальної моделі з найкращими гіперпараметрами
final_clf = XGBClassifier(
    n_estimators=best['n_estimators'],
    learning_rate=best['learning_rate'],
    max_depth=best['max_depth'],
    min_child_weight=best['min_child_weight'],
    subsample=best['subsample'],
    colsample_bytree=best['colsample_bytree'],
    gamma=best['gamma'],
    reg_alpha=best['reg_alpha'],
    reg_lambda=best['reg_lambda'],
    enable_categorical=True,
    use_label_encoder=False,
    missing=np.nan,
    device='cuda',
)

final_clf.fit(train_inputs, train_targets)

In [ ]:
display('Найкращі гіперпараметри: ', best)
print('')
get_auroc(final_clf, train_inputs, train_targets, 'Train')
get_auroc(final_clf, val_inputs, val_targets, 'Validation')

'Найкращі гіперпараметри: '

{'colsample_bytree': 0.9819568586853631,
 'gamma': 0.1157564882197677,
 'learning_rate': 0.09193610297543496,
 'max_depth': 5,
 'min_child_weight': 5,
 'n_estimators': 175,
 'reg_alpha': 0.36268163196126624,
 'reg_lambda': 0.7858716891341281,
 'subsample': 0.6374854520166109}


Train ROC AUC: 0.9669
Validation ROC AUC: 0.9333


This model shows better results for both train and validation data.   
It means that with more suitable parametres model show better results.

4. Навчіть на наших даних модель LightGBM. Параметри алгоритму встановіть на свій розсуд, ми далі будемо їх тюнити. Рекомендую тренувати не дуже складну модель.

  Опис всіх конфігураційних параметрів LightGBM - тут https://lightgbm.readthedocs.io/en/latest/Parameters.html

  **Важливо:** зробіть такі налаштування LightGBM аби він самостійно обробляв незаповнені значення в даних і обробляв категоріальні колонки.

  Аби передати категоріальні колонки в LightGBM - необхідно виявити їх індекси і передати в параметрі `cat_feature=cat_feature_indexes`

  Після тренування моделі
  1. Виміряйте точність з допомогою AUROC на тренувальному та валідаційному наборах.
  2. Зробіть висновок про отриману модель: вона хороша/погана, чи є high bias/high variance?
  3. Порівняйте якість цієї моделі з тою, що ви отрмали з використанням XGBoostClassifier раніше. Чи вийшло покращити якість?

In [ ]:
# %%bash
# sudo apt-get update
# sudo apt-get install -y build-essential cmake git wget unzip
# sudo apt-get install -y libboost-dev libboost-system-dev libboost-filesystem-dev
# sudo apt-get install -y libboost-iostreams-dev libboost-program-options-dev libboost-regex-dev
# sudo apt-get install -y libboost-thread-dev libboost-chrono-dev libboost-date-time-dev
# sudo apt-get install -y libboost-atomic-dev libboost-serialization-dev
# sudo apt-get install -y python3-pip

In [ ]:
# %%bash
# sudo apt-get install -y ocl-icd-libopencl1 clinfo
# sudo apt-get install -y nvidia-opencl-dev opencl-headers

In [ ]:
# %%bash
# git clone --recursive https://github.com/microsoft/LightGBM
# cd LightGBM
# mkdir build
# cd build
# cmake -DUSE_CUDA=1 ..
# make -j4

In [ ]:
import lightgbm as lgb
print(lgb.__version__)

4.6.0


In [ ]:
cat_feature_indexes = [train_inputs.columns.get_loc(col) for col in cat_features]

In [ ]:
lgb_clf = lgb.LGBMClassifier(
    max_depth=5,
    n_estimators=50,
    learning_rate=0.1,
    cat_feature=cat_feature_indexes,
    #device='cuda'  # використовувати GPU для прискорення обчислень
)

lgb_clf.fit(train_inputs, train_targets, eval_set=[(val_inputs, val_targets)])

In [ ]:
get_auroc(lgb_clf, train_inputs, train_targets, 'Train')
get_auroc(lgb_clf, val_inputs, val_targets, 'Validation')

Train ROC AUC: 0.9688
Validation ROC AUC: 0.9350


`LightGBM` showed better result than `XGBoostClassifier`.

5. Використовуючи бібліотеку `Hyperopt` і приклад пошуку гіперпараметрів для `LightGBM` з лекції знайдіть оптимальні значення гіперпараметрів `LightGBM` для нашої задачі. Задайте свою сітку гіперпараметрів виходячи з тих параметрів, які ви б хотіли перебрати. Поставте кількість раундів в підборі гіперпараметрів рівну **10**.

  **Увага!** Для того, аби скористатись hyperopt, нам треба задати функцію `objective`. І тут ми також ставимо loss - негативне значення AUROC, як і при пошуці гіперпараметрів для XGBoost. До речі, можна спробувати написати код так, аби в objective передавати лише модель і не писати схожий код двічі :)

  Після успішного завершення пошуку оптимальних гіперпараметрів
    - виведіть найкращі значення гіперпараметрів
    - створіть в окремій зміній `final_lgb_clf` модель `LightGBM` з найкращими гіперпараметрами
    - навчіть модель `final_lgb_clf`
    - оцініть якість моделі `final_lgb_clf` на тренувальній і валідаційній вибірках з допомогою AUROC.
    - зробіть висновок про якість моделі. Чи стала вона краще порівняно з попереднім пунктом (4) цього завдання?

In [ ]:
def objective(params):
    lgb_clf = lgb.LGBMClassifier(
        n_estimators=int(params['n_estimators']),  # Кількість дерев у ансамблі (кількість ітерацій бустингу)
        learning_rate=params['learning_rate'],  # Коефіцієнт, на який зменшується внесок кожного доданого дерева
        max_depth=int(params['max_depth']),  # Максимальна глибина кожного дерева
        num_leaves=int(params['num_leaves']),  # Максимальна кількість листків, що дозволяємо кожному дереву мати.
        min_child_weight=params['min_child_weight'],  # Мінімальна сума ваг всіх вибірок, необхідна в кінцевому вузлі
        subsample=params['subsample'],  # Частка вибірок, що використовуються для побудови кожного дерева
        colsample_bytree=params['colsample_bytree'],  # Частка ознак, що використовуються при побудові кожного дерева
        reg_alpha=params['reg_alpha'],  # Параметр регуляризації L1 (Lasso)
        reg_lambda=params['reg_lambda'],  # Параметр регуляризації L2 (Ridge)
        min_split_gain=params['min_split_gain'],  # Мінімальне зменшення втрат, необхідне для виконання поділу
        cat_feature=cat_feature_indexes  # Індекси категорійних ознак
    )

    lgb_clf.fit(train_inputs, train_targets, eval_set=[(val_inputs, val_targets)])
    pred = lgb_clf.predict_proba(val_inputs)[:, 1]
    fpr, tpr, _ = roc_curve(val_targets, pred)
    roc_auc = auc(fpr, tpr)
    return {'loss': -roc_auc, 'status': STATUS_OK}

# Простір гіперпараметрів
space = {
    'n_estimators': hp.quniform('n_estimators', 50, 500, 25),
    'learning_rate': hp.uniform('learning_rate', 0.01, 0.3),
    'max_depth': hp.quniform('max_depth', 3, 15, 1),
    'num_leaves': hp.quniform('num_leaves', 20, 150, 1),
    'min_child_weight': hp.quniform('min_child_weight', 1, 10, 1),
    'subsample': hp.uniform('subsample', 0.5, 1.0),
    'colsample_bytree': hp.uniform('colsample_bytree', 0.5, 1.0),
    'reg_alpha': hp.uniform('reg_alpha', 0, 1),
    'reg_lambda': hp.uniform('reg_lambda', 0, 1),
    'min_split_gain': hp.uniform('min_split_gain', 0, 0.1)  # додано мінімальне зменшення втрат для поділу
}

# Оптимізація
trials = Trials()
best = fmin(fn=objective, space=space, algo=tpe.suggest, max_evals=10, trials=trials)

# Перетворення значень гіперпараметрів у кінцеві типи
best['n_estimators'] = int(best['n_estimators'])
best['max_depth'] = int(best['max_depth'])
best['num_leaves'] = int(best['num_leaves'])
best['min_child_weight'] = int(best['min_child_weight'])


# Навчання фінальної моделі з найкращими гіперпараметрами
final_lgb_clf = lgb.LGBMClassifier(
    n_estimators=best['n_estimators'],
    learning_rate=best['learning_rate'],
    max_depth=best['max_depth'],
    num_leaves=best['num_leaves'],
    min_child_weight=best['min_child_weight'],
    subsample=best['subsample'],
    colsample_bytree=best['colsample_bytree'],
    reg_alpha=best['reg_alpha'],
    reg_lambda=best['reg_lambda'],
    min_split_gain=best['min_split_gain'],
    cat_feature=cat_feature_indexes
)

final_lgb_clf.fit(train_inputs, train_targets, eval_set=[(val_inputs, val_targets)])

In [ ]:
display('Найкращі гіперпараметри: ', best)
print('')

get_auroc(final_lgb_clf, train_inputs, train_targets, 'Train')
get_auroc(final_lgb_clf, val_inputs, val_targets, 'Validation')

'Найкращі гіперпараметри: '

{'colsample_bytree': 0.7793595223916057,
 'learning_rate': 0.09923267417974863,
 'max_depth': 5,
 'min_child_weight': 8,
 'min_split_gain': 0.06576183647948851,
 'n_estimators': 275,
 'num_leaves': 67,
 'reg_alpha': 0.9858165981204177,
 'reg_lambda': 0.4512395505044936,
 'subsample': 0.5765560993057253}


Train ROC AUC: 0.9905
Validation ROC AUC: 0.9297


Final `LightGBM` with tunned parametres showed better result for validation data, but difference between final and previous lightGBM is nor significant.

6. Оберіть модель з експериментів в цьому ДЗ і зробіть новий `submission` на Kaggle та додайте код для цього і скріншот скора на публічному лідерборді.
  
  **Напишіть коментар, чому ви обрали саме цю модель?**

  І я вас вітаю - це останнє завдання з цим набором даних 💪 На цьому етапі корисно проаналізувати, які моделі показали себе найкраще і подумати, чому.

In [ ]:
train_pred = final_clf.predict(train_inputs)
val_pred = final_clf.predict(val_inputs)

print(classification_report(train_targets, train_pred, digits=4))
print(classification_report(val_targets, val_pred, digits=4))

              precision    recall  f1-score   support

         0.0     0.9417    0.9702    0.9557      9558
         1.0     0.8676    0.7649    0.8131      2442

    accuracy                         0.9284     12000
   macro avg     0.9047    0.8676    0.8844     12000
weighted avg     0.9266    0.9284    0.9267     12000

              precision    recall  f1-score   support

         0.0     0.9216    0.9548    0.9379      2390
         1.0     0.7939    0.6820    0.7337       610

    accuracy                         0.8993      3000
   macro avg     0.8578    0.8184    0.8358      3000
weighted avg     0.8957    0.8993    0.8964      3000



In [ ]:
train_pred = final_lgb_clf.predict(train_inputs)
val_pred = final_lgb_clf.predict(val_inputs)

print(classification_report(train_targets, train_pred, digits=4))
print(classification_report(val_targets, val_pred, digits=4))

              precision    recall  f1-score   support

         0.0     0.9675    0.9893    0.9783      9558
         1.0     0.9542    0.8698    0.9100      2442

    accuracy                         0.9650     12000
   macro avg     0.9608    0.9296    0.9442     12000
weighted avg     0.9648    0.9650    0.9644     12000

              precision    recall  f1-score   support

         0.0     0.9212    0.9439    0.9324      2390
         1.0     0.7568    0.6836    0.7183       610

    accuracy                         0.8910      3000
   macro avg     0.8390    0.8138    0.8254      3000
weighted avg     0.8878    0.8910    0.8889      3000



In [ ]:
get_auroc(final_clf, val_inputs, val_targets, 'Validation')
get_auroc(final_lgb_clf, val_inputs, val_targets, 'Validation')

Validation ROC AUC: 0.9333
Validation ROC AUC: 0.9297


We can see that `final_lgb_clf` model is more accurate.

In [ ]:
test_raw_df = pd.read_csv("drive/MyDrive/Colab Notebooks/data/test.csv")
test_inputs = test_raw_df[input_cols].copy()
test_inputs[cat_features] = test_inputs[cat_features].astype('category')
test_prob = final_lgb_clf.predict_proba(test_inputs)[:,1]
test_raw_df['Exited'] = test_prob

In [ ]:
submission_raw_df = pd.read_csv("drive/MyDrive/Colab Notebooks/data/sample_submission.csv")
submission_raw_df['Exited'] = test_raw_df['Exited']
submission_raw_df['Exited'].head()

,Exited
0,0.018328
1,0.003591
2,0.107750
3,0.676592
4,0.014175


In [ ]:
submission_raw_df.to_csv('submission_lgb_clf.csv', index=False)